In [ ]:
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

epochs = 25
batch_size = 100
learning_rate = 0.001

CIFAR100_MEAN = [0.5071, 0.4865, 0.4409]
CIFAR100_STD = [0.2673, 0.2564, 0.2762]
DATA_ROOT = './cifar100_data'

train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD)
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD)
])

train_dataset = datasets.CIFAR100(root=DATA_ROOT, train=True, transform=train_transforms, download=True)
test_dataset = datasets.CIFAR100(root=DATA_ROOT, train=False, transform=test_transforms)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
def create_conv2d(in_channels, out_channels, kernel_size, stride=1, padding=1, groups=1):
    return nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride,
                     padding=padding, groups=groups)

In [ ]:
class SELayer(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels),
            nn.Hardsigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        out = self.avg_pool(x).view(b, c)
        out = self.fc(out).view(b, c, 1, 1)
        out = x * out.expand_as(x)

        return out

In [ ]:
class BneckBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, expansion_rate=6.0,
                 act_func=nn.ReLU(inplace=True), selayer=False, stride=1):
        super().__init__()
        expansion_channels = int(in_channels * expansion_rate)
        padding = kernel_size // 2
        self.shortcut_connection = True if (in_channels == out_channels) and (stride == 1) else False

        self.conv1 = create_conv2d(in_channels=in_channels, out_channels=expansion_channels, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(expansion_channels)
        self.act_func = act_func

        self.conv2 = create_conv2d(in_channels=expansion_channels, out_channels=expansion_channels,
                                   kernel_size=kernel_size, stride=stride, padding=padding, groups=expansion_channels)
        self.bn2 = nn.BatchNorm2d(expansion_channels)
        self.selayer = SELayer(in_channels=expansion_channels, reduction=2) if selayer else None

        self.conv3 = create_conv2d(in_channels=expansion_channels, out_channels=out_channels, kernel_size=1)
        self.bn3 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act_func(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.act_func(out)

        if self.selayer:
            out = self.selayer(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.shortcut_connection:
            out += residual

        return out

In [ ]:
class MobilenetV3(nn.Module):
    def __init__(self, block, num_classes=1000):
        super().__init__()
        self.in_channels = 16
        relu = nn.ReLU(inplace=True)
        h_swish = nn.Hardswish(inplace=True)

        bneck_blocks = [
            self.create_bneck_block(out_channels=16, kernel_size=3, expansion_rate=1, act_func=relu, selayer=False, stride=1),
            self.create_bneck_block(out_channels=24, kernel_size=3, expansion_rate=4, act_func=relu,                                   selayer=True, stride=2),
            self.create_bneck_block(out_channels=24, kernel_size=3, expansion_rate=3, act_func=relu,                                   selayer=False, stride=1),
            self.create_bneck_block(out_channels=40, kernel_size=5, expansion_rate=3, act_func=h_swish,                                selayer=True, stride=2),
            self.create_bneck_block(out_channels=40, kernel_size=5, expansion_rate=3, act_func=h_swish,                                selayer=True, stride=1),
            self.create_bneck_block(out_channels=40, kernel_size=5, expansion_rate=3, act_func=h_swish,                                selayer=True, stride=1),
            self.create_bneck_block(out_channels=80, kernel_size=3, expansion_rate=6, act_func=h_swish,                                selayer=False, stride=2),
            self.create_bneck_block(out_channels=80, kernel_size=3, expansion_rate=2.5, act_func=h_swish,                              selayer=False, stride=1),
            self.create_bneck_block(out_channels=80, kernel_size=3, expansion_rate=2.3, act_func=h_swish,                              selayer=True, stride=1),
            self.create_bneck_block(out_channels=112, kernel_size=3, expansion_rate=2.3, act_func=h_swish,
                                    selayer=False, stride=1),
            self.create_bneck_block(out_channels=112, kernel_size=3, expansion_rate=4.3, act_func=h_swish,                             selayer=True, stride=2),
            self.create_bneck_block(out_channels=160, kernel_size=3, expansion_rate=6, act_func=h_swish,                               selayer=True, stride=1),
            self.create_bneck_block(out_channels=160, kernel_size=3, expansion_rate=4.2, act_func=h_swish,                             selayer=True, stride=1),
            self.create_bneck_block(out_channels=160, kernel_size=3, expansion_rate=4.2, act_func=h_swish,                             selayer=True, stride=1)
        ]

        self.features = nn.Sequential(
            create_conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16),
            h_swish,
            *bneck_blocks,
            create_conv2d(160, 960, 1),
            nn.BatchNorm2d(960),
            h_swish,
            nn.AdaptiveAvgPool2d(1),
            create_conv2d(960, 1280, 1),
            h_swish,
            nn.Flatten(),
            nn.Linear(1280, num_classes)
        )

    def create_bneck_block(self, out_channels, kernel_size=3, expansion_rate=6.0,
                           act_func=nn.ReLU(inplace=True), selayer=False, stride=1):
        block = BneckBlock(in_channels=self.in_channels, out_channels=out_channels, kernel_size=kernel_size,
                           expansion_rate=expansion_rate, act_func=act_func, selayer=selayer,
                           stride=stride)

        self.in_channels = out_channels

        return block

    def forward(self, x):
        out = self.features(x)

        return out